In [4]:
import sys
sys.path.insert(0, "..")

import yaml, torch
import torch.nn.functional as F
import numpy as np
import pandas as pd

from src.retinocare.data.dataset import get_dataloaders
from src.retinocare.models.baseline_cnn import BaselineCNN
from src.retinocare.models.resnet_transfer import build_resnet18, build_efficientnet_b0
from src.retinocare.evaluation.metrics import compute_classification_report, expected_calibration_error

with open("../configs/train_config.yaml") as f:
    config = yaml.safe_load(f)

config["data"]["train_csv"] = "../" + config["data"]["train_csv"]
config["data"]["val_csv"] = "../" + config["data"]["val_csv"]
config["data"]["test_csv"] = "../" + config["data"]["test_csv"]
config["data"]["image_dir"] = "../" + config["data"]["image_dir"]

_, _, test_dl = get_dataloaders(config)
SEVERITY_LABELS = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]

In [5]:
def evaluate_checkpoint(model, checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, map_location="cpu"))
    model.eval()

    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for images, labels in test_dl:
            logits = model(images)
            probs = F.softmax(logits, dim=1)
            all_preds.extend(logits.argmax(dim=1).tolist())
            all_labels.extend(labels.tolist())
            all_probs.extend(probs.tolist())

    report = compute_classification_report(all_labels, all_preds, SEVERITY_LABELS)
    ece = expected_calibration_error(np.array(all_probs), all_labels, n_bins=10)

    return {
        "weighted_f1": report["weighted avg"]["f1-score"],
        "macro_f1": report["macro avg"]["f1-score"],
        "severe_f1": report["Severe"]["f1-score"],
        "proliferative_f1": report["Proliferative DR"]["f1-score"],
        "ece": ece,
    }

In [ ]:
results = {}
results["baseline_cnn"] = evaluate_checkpoint(BaselineCNN(num_classes=5), "../models/baseline_cnn.pt")
results["resnet18"] = evaluate_checkpoint(build_resnet18(num_classes=5, freeze_backbone=False), "../models/resnet18.pt")
results["efficientnet_b0"] = evaluate_checkpoint(build_efficientnet_b0(num_classes=5, freeze_backbone=False), "../models/efficientnet_b0.pt")

comparison_df = pd.DataFrame(results).T
comparison_df

In [ ]:
comparison_df.plot(y="weighted_f1", kind="bar", figsize=(6,4), color="#1F4E5F", legend=False)
import matplotlib.pyplot as plt
plt.title("Weighted F1 by model")
plt.ylabel("Weighted F1")
plt.tight_layout()
plt.savefig("../docs/model_comparison.png", dpi=150)
plt.show()